# Perturbation Evaluation Metrics

## Load Files

In [9]:
import glob
import os
import pandas as pd

# Base directory
base_path = "/home/jovyan/work/MST/results/DINOv3ViTB/evaluation_results"

# Metrics to include
metrics = ["deletion", "insertion", "negative"]

# Collect all CSV files under:
# evaluation_results/{metric}/csv_files/absolute/*.csv
csv_files = []

for metric in metrics:
    metric_path = os.path.join(
        base_path,
        metric,
        "csv_files",
        "absolute",
        "*.csv"
    )
    
    files = glob.glob(metric_path)
    csv_files.extend(files)

print(f"Found {len(csv_files)} files")

# Load and combine
dfs = []

for file in csv_files:
    df = pd.read_csv(file)

    # Add metadata
    df["file_name"] = os.path.basename(file)
    df["metric"] = file.split(os.sep)[-4]  # deletion/insertion/negative

    dfs.append(df)

# Combine all dataframes
df_all = pd.concat(dfs, ignore_index=True)

# Preview
print(df_all.shape)
df_all.head()

Found 75 files
(8700, 10)


,UID,dataset,model,xai_method,mode,predicted_class,auc,replacement,file_name,metric
0,02F4A1FB_left,ODELIA,DinoV2ClassifierSlice,slice_weighted_rollout,deletion,0,0.628240,gaussian_blur,deletion_slice_weighted_rollout_gaussian_blur.csv,deletion
1,02F4A1FB_right,ODELIA,DinoV2ClassifierSlice,slice_weighted_rollout,deletion,0,0.556623,gaussian_blur,deletion_slice_weighted_rollout_gaussian_blur.csv,deletion
2,08FEB48B_left,ODELIA,DinoV2ClassifierSlice,slice_weighted_rollout,deletion,2,0.416956,gaussian_blur,deletion_slice_weighted_rollout_gaussian_blur.csv,deletion
3,1647533552_left,ODELIA,DinoV2ClassifierSlice,slice_weighted_rollout,deletion,0,0.448901,gaussian_blur,deletion_slice_weighted_rollout_gaussian_blur.csv,deletion
4,1647533552_right,ODELIA,DinoV2ClassifierSlice,slice_weighted_rollout,deletion,0,0.367227,gaussian_blur,deletion_slice_weighted_rollout_gaussian_blur.csv,deletion


## Micro Average

In [10]:
micro_summary = (
    df_all.groupby(["mode", "xai_method", "replacement"])["auc"]
    .agg(["mean", "std", "count"])
    .reset_index()
)
micro_summary["mean"] = micro_summary["mean"].round(3)
micro_summary["std"] = micro_summary["std"].round(3)
print("=== Micro-average (overall) ===")
micro_summary

=== Micro-average (overall) ===


,mode,xai_method,replacement,mean,std,count
0,deletion,grad_sam,attention_mask,0.419,0.178,116
1,deletion,grad_sam,black-5,0.293,0.133,116
2,deletion,grad_sam,gaussian_blur,0.473,0.238,116
3,deletion,grad_sam,minimum-intensity,0.406,0.181,116
4,deletion,grad_sam,white-5,0.314,0.114,116
5,deletion,gradcam,attention_mask,0.426,0.196,116
6,deletion,gradcam,black-5,0.315,0.148,116
7,deletion,gradcam,gaussian_blur,0.416,0.197,116
8,deletion,gradcam,minimum-intensity,0.445,0.239,116
9,deletion,gradcam,white-5,0.290,0.118,116


## Class-wise

In [11]:
class_summary = (
    df_all.groupby(["mode", "xai_method", "replacement", "predicted_class"])["auc"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

class_summary["mean"] = class_summary["mean"].round(3)
class_summary["std"] = class_summary["std"].round(3)

print("=== Class-wise (macro view) ===")
# Source - https://stackoverflow.com/a/16433953
# Posted by Wouter Overmeire, modified by community. See post 'Timeline' for change history
# Retrieved 2026-04-25, License - CC BY-SA 4.0

pd.set_option('display.max_rows', 500)

class_summary


=== Class-wise (macro view) ===


,mode,xai_method,replacement,predicted_class,mean,std,count
0,deletion,grad_sam,attention_mask,0,0.514,0.108,73
1,deletion,grad_sam,attention_mask,1,0.386,0.099,21
2,deletion,grad_sam,attention_mask,2,0.134,0.075,22
3,deletion,grad_sam,black-5,0,0.349,0.127,73
4,deletion,grad_sam,black-5,1,0.210,0.082,21
5,deletion,grad_sam,black-5,2,0.184,0.072,22
6,deletion,grad_sam,gaussian_blur,0,0.620,0.149,73
7,deletion,grad_sam,gaussian_blur,1,0.167,0.055,21
8,deletion,grad_sam,gaussian_blur,2,0.280,0.158,22
9,deletion,grad_sam,minimum-intensity,0,0.518,0.115,73


In [12]:
class_summary[class_summary["mode"] == 'deletion']

,mode,xai_method,replacement,predicted_class,mean,std,count
0,deletion,grad_sam,attention_mask,0,0.514,0.108,73
1,deletion,grad_sam,attention_mask,1,0.386,0.099,21
2,deletion,grad_sam,attention_mask,2,0.134,0.075,22
3,deletion,grad_sam,black-5,0,0.349,0.127,73
4,deletion,grad_sam,black-5,1,0.210,0.082,21
5,deletion,grad_sam,black-5,2,0.184,0.072,22
6,deletion,grad_sam,gaussian_blur,0,0.620,0.149,73
7,deletion,grad_sam,gaussian_blur,1,0.167,0.055,21
8,deletion,grad_sam,gaussian_blur,2,0.280,0.158,22
9,deletion,grad_sam,minimum-intensity,0,0.518,0.115,73


## Macro Average

In [13]:
macro_summary = (
    class_summary
    .groupby(["mode", "xai_method", "replacement"])["mean"]
    .mean()
    .reset_index(name="macro_mean_auc")
)

print("=== Macro-average (class-balanced) ===")
macro_summary

=== Macro-average (class-balanced) ===


,mode,xai_method,replacement,macro_mean_auc
0,deletion,grad_sam,attention_mask,0.344667
1,deletion,grad_sam,black-5,0.247667
2,deletion,grad_sam,gaussian_blur,0.355667
3,deletion,grad_sam,minimum-intensity,0.315333
4,deletion,grad_sam,white-5,0.290000
5,deletion,gradcam,attention_mask,0.348333
6,deletion,gradcam,black-5,0.267333
7,deletion,gradcam,gaussian_blur,0.316333
8,deletion,gradcam,minimum-intensity,0.322667
9,deletion,gradcam,white-5,0.245333


In [14]:
macro_summary = (
    df_all
    .groupby(["mode", "xai_method", "replacement", "predicted_class"])["auc"]
    .mean()
    .groupby(["mode", "xai_method", "replacement"])
    .agg(macro_mean_auc="mean", macro_std_auc="std", count="count")
    .reset_index()
)
macro_summary["macro_mean_auc"] = macro_summary["macro_mean_auc"].round(3)
macro_summary["macro_std_auc"] = macro_summary["macro_std_auc"].round(3)
print("=== Macro-average (class-balanced) ===")
macro_summary

=== Macro-average (class-balanced) ===


,mode,xai_method,replacement,macro_mean_auc,macro_std_auc,count
0,deletion,grad_sam,attention_mask,0.345,0.194,3
1,deletion,grad_sam,black-5,0.248,0.089,3
2,deletion,grad_sam,gaussian_blur,0.356,0.235,3
3,deletion,grad_sam,minimum-intensity,0.315,0.176,3
4,deletion,grad_sam,white-5,0.290,0.103,3
5,deletion,gradcam,attention_mask,0.348,0.198,3
6,deletion,gradcam,black-5,0.267,0.107,3
7,deletion,gradcam,gaussian_blur,0.316,0.196,3
8,deletion,gradcam,minimum-intensity,0.323,0.240,3
9,deletion,gradcam,white-5,0.245,0.095,3


## Combine Micro and Macro average

In [15]:
replacement_order = [
    "black-5",
    "minimum-intensity",
    "white-5",
    "gaussian_blur",
    "attention_mask"
]

In [16]:
combined_summary = (
    micro_summary
    .rename(columns={"mean": "micro_mean_auc", "std": "micro_std_auc"})
    .merge(macro_summary, on=["mode", "xai_method", "replacement"])
)

print("=== Combined Summary ===")
combined_summary

=== Combined Summary ===


,mode,xai_method,replacement,micro_mean_auc,micro_std_auc,count_x,macro_mean_auc,macro_std_auc,count_y
0,deletion,grad_sam,attention_mask,0.419,0.178,116,0.345,0.194,3
1,deletion,grad_sam,black-5,0.293,0.133,116,0.248,0.089,3
2,deletion,grad_sam,gaussian_blur,0.473,0.238,116,0.356,0.235,3
3,deletion,grad_sam,minimum-intensity,0.406,0.181,116,0.315,0.176,3
4,deletion,grad_sam,white-5,0.314,0.114,116,0.290,0.103,3
5,deletion,gradcam,attention_mask,0.426,0.196,116,0.348,0.198,3
6,deletion,gradcam,black-5,0.315,0.148,116,0.267,0.107,3
7,deletion,gradcam,gaussian_blur,0.416,0.197,116,0.316,0.196,3
8,deletion,gradcam,minimum-intensity,0.445,0.239,116,0.323,0.240,3
9,deletion,gradcam,white-5,0.290,0.118,116,0.245,0.095,3


In [17]:
combined_summary["replacement"] = pd.Categorical(
    combined_summary["replacement"],
    categories=replacement_order,
    ordered=True
)

In [18]:
combined_summary = combined_summary.sort_values(
    by=["mode", "xai_method", "replacement"]
).reset_index(drop=True)

In [19]:
combined_summary

,mode,xai_method,replacement,micro_mean_auc,micro_std_auc,count_x,macro_mean_auc,macro_std_auc,count_y
0,deletion,grad_sam,black-5,0.293,0.133,116,0.248,0.089,3
1,deletion,grad_sam,minimum-intensity,0.406,0.181,116,0.315,0.176,3
2,deletion,grad_sam,white-5,0.314,0.114,116,0.290,0.103,3
3,deletion,grad_sam,gaussian_blur,0.473,0.238,116,0.356,0.235,3
4,deletion,grad_sam,attention_mask,0.419,0.178,116,0.345,0.194,3
5,deletion,gradcam,black-5,0.315,0.148,116,0.267,0.107,3
6,deletion,gradcam,minimum-intensity,0.445,0.239,116,0.323,0.240,3
7,deletion,gradcam,white-5,0.290,0.118,116,0.245,0.095,3
8,deletion,gradcam,gaussian_blur,0.416,0.197,116,0.316,0.196,3
9,deletion,gradcam,attention_mask,0.426,0.196,116,0.348,0.198,3


## Method Comparison

In [20]:
method_summary = (
    df_all.groupby(["mode", "xai_method"])["auc"]
    .agg(["mean", "std"])
    .reset_index()
)

print("=== Method comparison ===")
method_summary

=== Method comparison ===


,mode,xai_method,mean,std
0,deletion,grad_sam,0.380976,0.186351
1,deletion,gradcam,0.378708,0.194210
2,deletion,hires_cam,0.480312,0.232148
3,deletion,last_layer,0.431473,0.197616
4,deletion,slice_weighted_rollout,0.440078,0.198635
5,insertion,grad_sam,0.651726,0.198937
6,insertion,gradcam,0.684960,0.156245
7,insertion,hires_cam,0.635888,0.198971
8,insertion,last_layer,0.626326,0.202442
9,insertion,slice_weighted_rollout,0.660452,0.185519


## Strategy Comparison

In [21]:
strategy_summary = (
    df_all.groupby(["mode", "replacement"])["auc"]
    .agg(["mean", "std"])
    .reset_index()
)

print("=== Strategy comparison ===")
strategy_summary

=== Strategy comparison ===


,mode,replacement,mean,std
0,deletion,attention_mask,0.486166,0.202385
1,deletion,black-5,0.336209,0.168016
2,deletion,gaussian_blur,0.465210,0.230823
3,deletion,minimum-intensity,0.467846,0.224819
4,deletion,white-5,0.356116,0.139277
5,insertion,attention_mask,0.780533,0.140770
6,insertion,black-5,0.538934,0.168306
7,insertion,gaussian_blur,0.600190,0.161008
8,insertion,minimum-intensity,0.760448,0.143105
9,insertion,white-5,0.579247,0.193351


## Best Method

In [22]:
# For deletion → lower is better
best_deletion = micro_summary[micro_summary["mode"] == "deletion"].sort_values("mean")

# For insertion → higher is better
best_insertion = micro_summary[micro_summary["mode"] == "insertion"].sort_values("mean", ascending=False)

print("=== Best (Deletion) ===")
print(best_deletion.head())

print("\n=== Best (Insertion) ===")
print(best_insertion.head())

=== Best (Deletion) ===
        mode  xai_method replacement   mean    std  count
9   deletion     gradcam     white-5  0.290  0.118    116
1   deletion    grad_sam     black-5  0.293  0.133    116
4   deletion    grad_sam     white-5  0.314  0.114    116
6   deletion     gradcam     black-5  0.315  0.148    116
16  deletion  last_layer     black-5  0.321  0.150    116

=== Best (Insertion) ===
         mode              xai_method        replacement   mean    std  count
30  insertion                 gradcam     attention_mask  0.795  0.111    116
45  insertion  slice_weighted_rollout     attention_mask  0.793  0.129    116
25  insertion                grad_sam     attention_mask  0.789  0.155    116
33  insertion                 gradcam  minimum-intensity  0.785  0.103    116
35  insertion               hires_cam     attention_mask  0.764  0.147    116


## Save Everythings

In [23]:
micro_summary.to_csv("micro_summary.csv", index=False)
class_summary.to_csv("class_summary.csv", index=False)
macro_summary.to_csv("macro_summary.csv", index=False)
method_summary.to_csv("method_summary.csv", index=False)
strategy_summary.to_csv("strategy_summary.csv", index=False)

print("All results saved")

All results saved


# Calculate curve again without Normalized Confidence

In [24]:
evaluation_methods = {
    "deletion": "Deletion",
    "insertion": "Insertion",
    "negative": "Negative Perturbation",
}

replacement_strategies = {
    "black-5": "Black masking", 
    "minimum-intensity": "Minimum intensity",
    "white-5": "White masking", 
    "gaussian_blur": "Gaussian Blur",
    "attention_mask" : "Attention Mask",  
}

xai_methods = {
    "gradcam": "Grad-CAM",
    "last_layer": "Last-layer attention", 
    "slice_weighted_rollout": "Attention rollout" 
}

colors = {
    "gradcam": "#1f77b4",              # blue
    "last_layer": "#ff7f0e",           # orange
    "slice_weighted_rollout": "#2ca02c"  # green
}

In [25]:
import numpy as np
from pathlib import Path
base_path = Path("/home/jovyan/work/MST/results/DINOv3ViTB/evaluation_results")
test_data = np.load("/home/jovyan/work/MST/results/DINOv3ViTB/evaluation_results/deletion/deletion_curves/gradcam/attention_mask/02F4A1FB_left_curve_attention_mask.npy", allow_pickle=True).item()
test_data.keys()

dict_keys(['percentages', 'raw_logits', 'confidences', 'normalized_confidences', 'predicted_class', 'predicted_confidences', 'predicted_normalized_confidences', 'auc', 'replacement'])